In [1]:

!pip install -q -U gradio langchain-groq langchain-community langchain-text-splitters langchain-huggingface faiss-cpu pypdf sentence-transformers


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.4/31.4 MB 38.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.5/64.5 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 61.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 39.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 402.7/402.7 kB 19.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 740.6/740.6 kB 32.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 50.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 5.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests=

In [2]:
import os
import getpass

GROQ_API_KEY = getpass.getpass("Enter your Groq API key: ")
os.environ["GROQ_API_KEY"] = GROQ_API_KEY

Enter your Groq API key: ··········


In [3]:

import gradio as gr

from langchain_groq import ChatGroq
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# -----------------------------
# 1. Initialize the language model
# -----------------------------
llm = ChatGroq(
    api_key=GROQ_API_KEY,
    model="openai/gpt-oss-120b",
    temperature=0
)

# -----------------------------
# 2. Initialize embeddings
# Downloads the embedding model the first time
# -----------------------------
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

# -----------------------------
# 3. Create the RAG prompt
# -----------------------------
prompt = ChatPromptTemplate.from_template("""
You are a helpful PDF question-answering assistant.

Answer using only the information in the context.
If the answer is not available in the context, say:
"I couldn't find that information in this PDF."

Context:
{context}

Question:
{question}

Answer:
""")

# -----------------------------
# 4. Process uploaded PDF
# -----------------------------
def process_pdf(pdf_file):
    if pdf_file is None:
        return "🌷 Please upload a PDF first."

    try:
        loader = PyPDFLoader(pdf_file)
        documents = loader.load()

        if not documents:
            return "⚠️ No readable text was found in this PDF."

        splitter = RecursiveCharacterTextSplitter(
            chunk_size=800,
            chunk_overlap=120
        )

        chunks = splitter.split_documents(documents)

        if not chunks:
            return "⚠️ No text chunks could be created."

        global vectorstore, retriever

        vectorstore = FAISS.from_documents(
            chunks,
            embeddings
        )

        retriever = vectorstore.as_retriever(
            search_kwargs={"k": 3}
        )

        return (
            f"✅ PDF processed successfully!\n\n"
            f"📄 Pages loaded: {len(documents)}\n"
            f"🧩 Text chunks created: {len(chunks)}\n\n"
            "💖 You can now ask questions about your PDF!"
        )

    except Exception as e:
        return f"⚠️ Could not process PDF: {str(e)}"

# -----------------------------
# 5. Answer questions using RAG
# -----------------------------
def answer_question(question):
    if not question or not question.strip():
        return "🌸 Please type a question!", ""

    if "retriever" not in globals():
        return "📄 Please upload and process a PDF first.", ""

    try:
        docs = retriever.invoke(question)

        if not docs:
            return "I couldn't find relevant information.", ""

        context = "\n\n".join(
            doc.page_content for doc in docs
        )

        rag_chain = prompt | llm | StrOutputParser()

        answer = rag_chain.invoke({
            "context": context,
            "question": question
        })

        sources = "\n\n".join(
            f"📖 Page {doc.metadata.get('page', 0) + 1}\n"
            f"{doc.page_content}"
            for doc in docs
        )

        return f"✨ {answer}", sources

    except Exception as e:
        return f"⚠️ Error: {str(e)}", ""

# -----------------------------
# 6. Cute pastel design
# -----------------------------
custom_css = """
.gradio-container {
    background: linear-gradient(
        135deg, #fff0f6, #f0edff, #e6fff5
    ) !important;
}

#main-title {
    text-align: center;
    color: #8055a5;
}

#main-title h1 {
    font-size: 34px;
}

.panel {
    background: rgba(255,255,255,0.85) !important;
    border: 2px solid #eadcfb !important;
    border-radius: 20px !important;
    padding: 18px !important;
}

#ask-button {
    background: linear-gradient(
        90deg, #b69afa, #f3a4cf
    ) !important;
    color: white !important;
    border: none !important;
    border-radius: 14px !important;
    font-weight: bold !important;
}
"""

# -----------------------------
# 7. Build the Gradio app
# -----------------------------
with gr.Blocks(
    theme=gr.themes.Soft(
        primary_hue="purple",
        secondary_hue="pink"
    ),
    css=custom_css,
    title="Cute PDF RAG Assistant"
) as demo:

    gr.Markdown(
        """
        # 🎀 My PDF Study Buddy 📚
        ### Upload a PDF, ask anything, and learn something new! ✨
        """,
        elem_id="main-title"
    )

    with gr.Row():
        with gr.Column(elem_classes=["panel"]):
            gr.Markdown("## 📄 Upload Your PDF")

            pdf_input = gr.File(
                label="Choose a PDF file",
                file_types=[".pdf"],
                type="filepath"
            )

            upload_button = gr.Button(
                "🌷 Process My PDF",
                variant="primary"
            )

            status_box = gr.Textbox(
                label="💌 PDF Status",
                lines=4,
                interactive=False
            )

        with gr.Column(elem_classes=["panel"]):
            gr.Markdown("## 💭 Ask Your PDF")

            question_input = gr.Textbox(
                label="Your question",
                placeholder="What is this PDF about?",
                lines=3
            )

            ask_button = gr.Button(
                "✨ Find My Answer",
                elem_id="ask-button"
            )

            answer_output = gr.Textbox(
                label="💖 Answer",
                lines=7,
                interactive=False
            )

    with gr.Accordion("📚 View Retrieved PDF Text", open=False):
        sources_output = gr.Textbox(
            label="Relevant text and page numbers",
            lines=12,
            interactive=False
        )

    upload_button.click(
        fn=process_pdf,
        inputs=pdf_input,
        outputs=status_box
    )

    ask_button.click(
        fn=answer_question,
        inputs=question_input,
        outputs=[answer_output, sources_output]
    )

    question_input.submit(
        fn=answer_question,
        inputs=question_input,
        outputs=[answer_output, sources_output]
    )

    gr.Markdown(
        "<center>🌸 Made with Python + RAG + Gradio 💜</center>"
    )

demo.launch(share=True)


/tmp/ipykernel_4070/3960592881.py:4: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

/tmp/ipykernel_4070/3960592881.py:170: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme, css. Please pass these parameters to launch() instead.
  with gr.Blocks(


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://82fdaa8ec032f833a0.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
